# 15 Generated Sample Grid — Baseline vs SDD

Visual comparison of samples drawn from the baseline (MSE-only) and full SDD model
using the **same random seed** for a fair side-by-side comparison.

Saved to `outputs/samples/comparison_grid.png` — ready to paste into a README or slides.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from src.experiments import (
    load_cfg, deep_update,
    build_trainer_from_checkpoint,
    generate_sample_grid, generate_comparison_grid,
)

cfg_baseline = load_cfg('configs/cifar10.yaml')
cfg_baseline = deep_update(cfg_baseline, {
    'run_name': 'cifar10_mse_only',
    'sdd': {'enabled': False, 'use_centering': False, 'use_sharpening': False,
            'use_gating': False, 'use_projection_head': False, 'lambda_distill': 0.0},
})

cfg_sdd = load_cfg('configs/cifar10.yaml')
cfg_sdd = deep_update(cfg_sdd, {'run_name': 'cifar10_full_sdd'})

# Load from checkpoints (train first with notebooks 02 and 03)
ckpt_baseline = 'outputs/checkpoints/cifar10_mse_only_last.pt'
ckpt_sdd      = 'outputs/checkpoints/cifar10_full_sdd_last.pt'

trainer_baseline = build_trainer_from_checkpoint(cfg_baseline, ckpt_baseline)
trainer_sdd      = build_trainer_from_checkpoint(cfg_sdd, ckpt_sdd)

In [ ]:
# Individual 8×8 grids for each model
generate_sample_grid(trainer_baseline, cfg_baseline, n=64, seed=42,
                     save_path='outputs/samples/baseline_grid.png')
generate_sample_grid(trainer_sdd, cfg_sdd, n=64, seed=42,
                     save_path='outputs/samples/sdd_grid.png')

# Side-by-side comparison grid (8 samples per model, one row each)
comparison_path = generate_comparison_grid(
    trainers={'MSE only (baseline)': trainer_baseline, 'Full SDD': trainer_sdd},
    cfgs={'MSE only (baseline)': cfg_baseline, 'Full SDD': cfg_sdd},
    n_per_variant=8,
    seed=42,
    save_path='outputs/samples/comparison_grid.png',
)
print('Saved:', comparison_path)

In [ ]:
# Display in notebook
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, path, title in zip(axes,
                            ['outputs/samples/baseline_grid.png', 'outputs/samples/sdd_grid.png'],
                            ['Baseline (MSE only)', 'Full SDD']):
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=13)
    ax.axis('off')
plt.suptitle('Generated samples — same seed (42)', fontsize=14)
plt.tight_layout()
plt.show()